# Joint-embedding MoE continual-learning experiments

This notebook runs the pre-registered joint-embedding protocol. Cross-validation remains continual learning: each raw CIL task is stratified into three folds; corresponding task folds are held out together. Final replicas train task-by-task on all selected `train.csv` records. No training or validation prediction caches are saved.

The only model-family difference is the pre-registered joint-embedding architecture and its supervised contrastive objective. Conditions use JE MobileNetV3 Large or JE ConvNeXt Tiny with a single-layer regression router or deeper MLP router.

In [ ]:
! git clone "https://github.com/ddimpfel/JHU_IS_26.git"
! cd "JHU_IS_26"

In [ ]:
import pandas as pd
from IPython.display import display

from init_experiment import (
    ExperimentConfig,
    ExperimentRunner,
    build_gtsrb_data,
    build_model_specs,
    environment_summary,
    prepare_notebook_runtime,
)

# Runtime and output configuration
RESULTS_NAMESPACE = 'joint_embedding_experiments'
RUN_NAME = 'joint_embedding_cil_protocol'  # Stable directory name; change only to begin a new run.
RESUME = True
RUN_SEEDS = (7,)  # Legacy fixed-split compatibility only.
FINAL_RUN_SEEDS = (7, 17, 27)

runtime = prepare_notebook_runtime(RESULTS_NAMESPACE)

# Reproducibility and data
SEED = 7
DATASET_NAME = 'meowmeowmeowmeowmeow/gtsrb-german-traffic-sign'
CLASS_IDS = (1, 2, 3, 4, 5, 7, 8, 9, 10)
VALIDATION_FRACTION = 0.20
IMAGE_SIZE = 224
BATCH_SIZE = 256
NUM_WORKERS = 4 if runtime.in_colab else 0
PIN_MEMORY = runtime.in_colab
PERSISTENT_WORKERS = runtime.in_colab

# Continual-learning optimization
NUM_TASKS = 3
CV_FOLDS = 3
CV_REPEATS = 1
CV_EPOCHS = 4
EPOCHS = 8  # Final full-source CIL epochs per task.
THROUGHPUT_WARMUP_BATCHES = 3
EXEMPLAR_RATIO = 0.065
TASK_1_LR = 0.001
LATER_TASK_LR_FACTOR = 0.1
USE_CLASS_MASKING = True
KD_TEMPERATURE = 2.0
LAMBDA_KD = 0.5

# Mixture-of-experts architecture
NUM_EXPERTS = 6
HIDDEN_EXPERT_SIZE = 128
DROPOUT = 0.1
TOP_K = 2
LAMBDA_AUX = 0.05
TRANSFORMER_D_MODEL = 32
TRANSFORMER_NHEAD = 4

# Joint-embedding architecture and objective
JE_EMBEDDING_DIM = 256
JE_PROJECTION_DIM = 256
JE_FEATURE_KEY = 'projections'  # 'embeddings' or 'projections'
JE_TEMPERATURE = 0.07
JE_CONTRASTIVE_WEIGHT = 0.1

# Execution
PRETRAINED_BACKBONES = True
DEVICE = None  # Set to 'cuda' or 'cpu' to override automatic selection.

config = ExperimentConfig(
    seed=SEED,
    run_seeds=RUN_SEEDS,
    cv_folds=CV_FOLDS,
    cv_repeats=CV_REPEATS,
    cv_epochs=CV_EPOCHS,
    final_run_seeds=FINAL_RUN_SEEDS,
    throughput_warmup_batches=THROUGHPUT_WARMUP_BATCHES,
    dataset_name=DATASET_NAME,
    class_ids=CLASS_IDS,
    validation_fraction=VALIDATION_FRACTION,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    num_tasks=NUM_TASKS,
    epochs=EPOCHS,
    exemplar_ratio=EXEMPLAR_RATIO,
    task_1_lr=TASK_1_LR,
    later_task_lr_factor=LATER_TASK_LR_FACTOR,
    use_class_masking=USE_CLASS_MASKING,
    kd_temperature=KD_TEMPERATURE,
    lambda_kd=LAMBDA_KD,
    num_experts=NUM_EXPERTS,
    hidden_expert_size=HIDDEN_EXPERT_SIZE,
    dropout=DROPOUT,
    top_k=TOP_K,
    lambda_aux=LAMBDA_AUX,
    transformer_d_model=TRANSFORMER_D_MODEL,
    transformer_nhead=TRANSFORMER_NHEAD,
    je_embedding_dim=JE_EMBEDDING_DIM,
    je_projection_dim=JE_PROJECTION_DIM,
    je_feature_key=JE_FEATURE_KEY,
    je_temperature=JE_TEMPERATURE,
    je_contrastive_weight=JE_CONTRASTIVE_WEIGHT,
    pretrained_backbones=PRETRAINED_BACKBONES,
    device=DEVICE,
 )

In [ ]:
data = build_gtsrb_data(config)
runner = ExperimentRunner(config, data, runtime.results_dir, run_name=RUN_NAME, resume=RESUME)

In [ ]:
display(pd.DataFrame([environment_summary(config)]))
display(pd.DataFrame({'Task': range(1, config.num_tasks + 1), 'Raw GTSRB classes': data.task_classes}))

In [ ]:
je_specs = build_model_specs(config, family='joint_embedding')
specification_df = pd.DataFrame([spec.__dict__ for spec in je_specs.values()])
display(specification_df)
print(f'JE feature space: {config.je_feature_key}')
print(f'Contrastive-loss weight: {config.je_contrastive_weight}')

## CIL cross-validation

For every fold, each CIL task trains on its two local training shards and validates on its corresponding held-out shard. The runner adds contrastive loss only to registered joint-embedding conditions and does not save CV checkpoints.

In [ ]:
cv_run = runner.run_joint_embedding_cross_validation(verbose=False)
cv_comparison_df = cv_run.comparison_df
cv_comparison_df.head()

In [ ]:
cv_summary_columns = [
    'Model', 'CV Repeat', 'CV Fold', 'Backbone', 'Router', 'Expert', 'Feature Space',
    'AvgAcc Micro F1', 'Backward Transfer Micro F1',
    'Forward Transfer Micro F1', 'Average Forgetting Micro F1',
    'Full Validation Micro F1', 'Validation ECE',
    'Validation Normalized Router Entropy', 'Validation Expected Expert Calls', 'Training Throughput (samples/s)', 'Validation Cost Proxy',
]
cv_summary_df = cv_comparison_df.loc[:, [column for column in cv_summary_columns if column in cv_comparison_df.columns]]
cv_summary_df.sort_values(['AvgAcc Micro F1', 'Full Validation Micro F1'], ascending=False).reset_index(drop=True)

## Final full-source CIL replicas

Run after the CV selection decision is fixed. This stage saves one checkpoint per selected condition and final seed.

In [ ]:
final_run = runner.run_joint_embedding_final(verbose=False)
final_comparison_df = final_run.comparison_df
final_comparison_df.head()

In [ ]:
final_summary_columns = [
    'Model', 'Final Seed', 'Backbone', 'Router', 'Expert', 'Feature Space',
    'Training Throughput (samples/s)', 'Training Cost Proxy', 'Num Parameters',
]
final_summary_df = final_comparison_df.loc[:, [column for column in final_summary_columns if column in final_comparison_df.columns]]
final_summary_df.sort_values(['Model', 'Final Seed']).reset_index(drop=True)